In [ ]:
# ==========================================
# STEP 1: INSTALL
# ==========================================

!pip install -q ultralytics

from google.colab import files
import zipfile
import os
import shutil
import random
import torch
from collections import Counter
from ultralytics import YOLO

print("Upload all_images.zip and all_labels.zip")

uploaded = files.upload()

# ==========================================
# STEP 2: EXTRACT
# ==========================================

IMAGE_ZIP = "all_images.zip"
LABEL_ZIP = "all_labels.zip"

IMAGE_EXTRACT = "/content/images_raw"
LABEL_EXTRACT = "/content/labels_raw"

os.makedirs(IMAGE_EXTRACT, exist_ok=True)
os.makedirs(LABEL_EXTRACT, exist_ok=True)

with zipfile.ZipFile(
    IMAGE_ZIP,
    "r"
) as z:

    z.extractall(
        IMAGE_EXTRACT
    )

with zipfile.ZipFile(
    LABEL_ZIP,
    "r"
) as z:

    z.extractall(
        LABEL_EXTRACT
    )

print("Extraction complete")

# ==========================================
# STEP 3: YOLO FOLDERS
# ==========================================

BASE="/content/DSVS"

folders=[

"train/images",
"train/labels",

"valid/images",
"valid/labels",

"test/images",
"test/labels"

]

for f in folders:

    os.makedirs(

        os.path.join(
            BASE,
            f
        ),

        exist_ok=True
    )

# ==========================================
# STEP 4: FIND IMAGES
# ==========================================

all_images=[]

for root,dirs,files_list in os.walk(
    IMAGE_EXTRACT
):

    for file in files_list:

        if file.lower().endswith(

            (
                ".jpg",
                ".jpeg",
                ".png"
            )

        ):

            all_images.append(

                os.path.join(
                    root,
                    file
                )

            )

print(
    "Total images:",
    len(all_images)
)

# ==========================================
# FIND LABEL
# ==========================================

def find_label(
        label_name
):

    for root,dirs,files in os.walk(
        LABEL_EXTRACT
    ):

        if label_name in files:

            return os.path.join(
                root,
                label_name
            )

    return None

# ==========================================
# STEP 5: OVERSAMPLE SUNGLASSES ×8
# ==========================================

boost=[]

for img_path in all_images:

    image_name=os.path.basename(
        img_path
    )

    label_name=(

        os.path.splitext(
            image_name
        )[0]

        +

        ".txt"

    )

    label_path=find_label(
        label_name
    )

    if label_path is None:
        continue

    with open(
        label_path,
        "r"
    ) as f:

        lines=f.readlines()

    has_sunglasses=False

    for line in lines:

        if line.strip():

            cls=int(
                line.split()[0]
            )

            if cls==2:

                has_sunglasses=True
                break

    if has_sunglasses:

        boost.extend(
            [img_path]*8
        )

all_images.extend(
    boost
)

print(
"\nSunglasses oversampling complete"
)

print(
"Extra images:",
len(boost)
)

print(
"Final pool:",
len(all_images)
)

# ==========================================
# STEP 6: SPLIT
# ==========================================

random.seed(42)

random.shuffle(
    all_images
)

train_size=int(
    len(all_images)*0.8
)

valid_size=int(
    len(all_images)*0.1
)

train_files=all_images[
:train_size
]

valid_files=all_images[
train_size:
train_size+valid_size
]

test_files=all_images[
train_size+valid_size:
]

print(
"Train:",
len(train_files)
)

print(
"Valid:",
len(valid_files)
)

print(
"Test:",
len(test_files)
)

# ==========================================
# STEP 7: COPY DATA
# ==========================================

def copy_split(
        image_list,
        split
):

    copied=0
    missing=0

    for img_path in image_list:

        image_name=os.path.basename(
            img_path
        )

        label_name=(

            os.path.splitext(
                image_name
            )[0]

            +

            ".txt"

        )

        label_path=find_label(
            label_name
        )

        shutil.copy(

            img_path,

            os.path.join(

                BASE,
                split,
                "images",
                image_name

            )

        )

        if label_path:

            shutil.copy(

                label_path,

                os.path.join(

                    BASE,
                    split,
                    "labels",
                    label_name

                )

            )

            copied+=1

        else:

            missing+=1

    print(
        split,
        "labels copied:",
        copied,
        "missing:",
        missing
    )


copy_split(
train_files,
"train"
)

copy_split(
valid_files,
"valid"
)

copy_split(
test_files,
"test"
)

# ==========================================
# STEP 8: CLASS DISTRIBUTION
# ==========================================

counter=Counter()

for split in [

"train",
"valid",
"test"

]:

    folder=os.path.join(

        BASE,
        split,
        "labels"

    )

    for file in os.listdir(
        folder
    ):

        with open(

            os.path.join(
                folder,
                file
            )

        ) as f:

            for line in f:

                if line.strip():

                    cls=int(
                        line.split()[0]
                    )

                    counter[
                        cls
                    ]+=1

print("\n===== CLASS COUNT =====")

names={

0:"phone",
1:"face",
2:"sunglasses"

}

for i in range(3):

    print(
        names[i],
        ":",
        counter[i]
    )

# ==========================================
# STEP 9: DATA YAML
# ==========================================

yaml="""
train: /content/DSVS/train/images
val: /content/DSVS/valid/images
test: /content/DSVS/test/images

nc: 3

names:
  0: phone
  1: face
  2: sunglasses
"""

yaml_path="/content/DSVS/data.yaml"

with open(
yaml_path,
"w"
) as f:

    f.write(
        yaml
    )

# ==========================================
# STEP 10: DEVICE
# ==========================================

device=0 if torch.cuda.is_available() else "cpu"

print(
"\nTraining device:",
device
)

# ==========================================
# STEP 11: TRAIN
# ==========================================

model = YOLO(
    "yolov8n.pt"
)

results = model.train(

    data=yaml_path,

    epochs=200,

    imgsz=640,

    batch=16,

    device=device,

    patience=50,

    workers=2,

    project="/content/DSVS_training",

    name="driver_monitoring",

    degrees=10,

    translate=0.05,

    scale=0.5,

    shear=2,

    fliplr=0.5,

    mosaic=0.7,

    mixup=0.15,

    copy_paste=0.4
)

# ==========================================
# STEP 12: VALIDATE
# ==========================================

metrics=model.val()

print(
"\nPer class mAP:"
)

print(
metrics.box.maps
)

# ==========================================
# STEP 13: DOWNLOAD
# ==========================================

best=(

"/content/DSVS_training/"
"driver_monitoring/"
"weights/best.pt"

)

print(
"\nBEST MODEL:"
)

print(
best
)

files.download(
best
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Upload all_images.zip and all_labels.zip


Saving all_images.zip to all_images.zip
Saving all_labels.zip to all_labels.zip
Extraction complete
Total images: 1011

Sunglasses oversampling complete
Extra images: 304
Final pool: 1315
Train: 1052
Valid: 131
Test: 132
train labels copied: 1012 missing: 40
valid labels copied: 129 missing: 2
test labels copied: 129 missing: 3

===== CLASS COUNT =====
phone : 1049
face : 431
sunglasses : 89

Training device: 0
Ultralytics 8.4.77 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.4, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/DSVS/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=Fa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!yolo export model="/content/DSVS_training/driver_monitoring/weights/best.pt" format=onnx opset=12 imgsz=320

Ultralytics 8.4.77 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/DSVS_training/driver_monitoring/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 608ms
Prepared 4 packages in 2.10s
Installed 4 packages in 270ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 3.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: 

In [ ]:
!pip uninstall -y onnxruntime-gpu
!pip install onnxruntime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

m = YOLO("/content/best.onnx")

print(m.names)